# Deploying an Agent: ShopAssist Service

**Exercise objective:** wrap a small e-commerce agent into a real HTTP service. The agent itself is a stub - the focus is the deployment machinery around it: input validation, session management, tool-call hygiene, an async-friendly FastAPI server with timeouts and background tasks, and the container + secret-management artefacts (`Dockerfile`, `.env`, `.gitignore`, `docker-compose.yml`) that let it run as a service rather than as a notebook.

## Where this sits in the course

Module 07 (*Deploy di un agente*) is the last module of the agentic-AI track. The theory slides cover four production concerns:

1. **Stateless vs stateful**: an HTTP service is stateless by default, but a multi-turn agent needs cross-request state - typically session management.
2. **Tools in production**: input sanitisation, logging, no hardcoded credentials, configuration via environment variables.
3. **Container as a thinking environment**: a `Dockerfile` is the way to lock a Python service's dependencies, OS, and entry point.
4. **Secrets and environment variables**: `.env` files plus `.gitignore` plus `.env.example` are the floor of secret hygiene.

The practice notebooks cover the first and third in isolation. This exercise stitches all four together on a small agent.

## Comparison with the rest of module 03

Exercises 01-06 were about *what an agent does*: tool calling, ReAct, Reflexion, memory. This one is about *how you ship it*. The agent function in step 3 is deliberately trivial - a stub that returns a synthetic reply with a fake delay if the message mentions "report" or "analysis". Swapping the stub for any of the agents from exercises 02-06 is a one-function change; everything else (validation, sessions, timeouts, background tasks, container, secrets) is reusable infrastructure.

| | Module 03 ex 01-06 | This exercise |
|---|---------------------|---------------|
| Focus | Agent behaviour | Service wrapper |
| LLM in the loop | Yes | No (stub) |
| Output | Notebook cells | HTTP endpoints + container |
| Reusability | Each agent is bespoke | Wrapper fits any agent function |

## What we build

Four steps that build on each other:

1. **Models and sessions.** Pydantic validators on the request shape; an in-memory `SessionManager` for cross-turn state.
2. **A safe tool.** Input sanitisation, logging, environment-variable config, no hardcoded secrets.
3. **Production server.** FastAPI app with `/chat` (timeout-gated), `/history`, `/report` (async background task), `/report/status`, `/health`.
4. **Containerisation.** `Dockerfile`, `requirements.txt`, `.env` and `.env.example`, `.gitignore`, `docker-compose.yml`. None of these files are valuable individually; together they are the deployment package.

> **No LLM dependency.** Unlike exercises 02-06, this notebook does not call any model. The agent stub keeps the focus on the service mechanics.

---
## 1. Setup

Five third-party packages: `fastapi` + `uvicorn` for the HTTP service, `pydantic` for typed request models, `httpx` for the in-notebook end-to-end test, `nest_asyncio` to allow `uvicorn` to share the Jupyter event loop. Standard-library `logging` is configured globally; everything that runs inside the agent or the tool uses the `shopassist` logger so production traces would carry a stable channel.

> **`nest_asyncio` is a notebook accommodation.** It lets us run a real FastAPI server from inside a Jupyter cell so the end-to-end test in step 3 can hit it via HTTP. In a real deployment you would not import `nest_asyncio` at all - you would run `uvicorn agent:app --host 0.0.0.0 --port 8000` from the shell (or, more realistically, from inside the container).

In [1]:
# Install once if needed:
# %pip install fastapi uvicorn nest-asyncio httpx pydantic

import asyncio
import logging
import os
import threading
import time
import uuid
from pathlib import Path
from typing import Optional

import httpx
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, field_validator

nest_asyncio.apply()
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("shopassist")

---
## Step 1 - Request model and session manager

Two pieces of plumbing that the rest of the service rides on.

### `ChatRequest`

A Pydantic model with three fields (`user_id`, `session_id`, `message`) and two validators. The validators look small but they are doing real defensive work that you do not want repeated across every handler:

- **Strip and reject empty `message`.** A `"   "` user input should not propagate to the agent; rejecting it at the boundary saves an LLM call and produces a cleaner error message.
- **Truncate `message` to 500 characters.** A hard upper bound on input size protects the downstream model (no surprise context-window blowups) and the database (no unbounded rows). 500 is a reasonable conversational ceiling; a real deployment would tune it per use case.
- **Reject empty `user_id`.** Without a user identifier, downstream session lookup, rate limiting, and audit logging all break.

### `SessionManager`

A trivial in-memory store keyed by session id. Three operations: create a new session and return its id, check whether a session exists, append a message to a session, fetch the history. The simplicity is deliberate - the *interface* is what the rest of the code depends on, the in-memory implementation is the simplest substitute. In production this would be backed by Redis or sqlite (more on this in the critical analysis); the public methods would not change.

In [2]:
# ── Request model with input validators ───────────────────────────────────────

class ChatRequest(BaseModel):
    user_id: str
    session_id: Optional[str] = None
    message: str

    @field_validator("message")
    @classmethod
    def validate_message(cls, value: str) -> str:
        cleaned = value.strip()
        if not cleaned:
            raise ValueError("message cannot be empty")
        return cleaned[:500]  # hard upper bound on input size

    @field_validator("user_id")
    @classmethod
    def validate_user_id(cls, value: str) -> str:
        cleaned = value.strip()
        if not cleaned:
            raise ValueError("user_id cannot be empty")
        return cleaned


# ── In-memory session manager ─────────────────────────────────────────────────

class SessionManager:
    # Minimal store: session_id -> list of {role, text}. Single-instance only;
    # replace with Redis or sqlite for any deployment that runs more than one
    # worker or that needs to survive a restart.

    def __init__(self) -> None:
        self._store: dict[str, list[dict]] = {}

    def create(self) -> str:
        sid = str(uuid.uuid4())[:8]
        self._store[sid] = []
        return sid

    def exists(self, sid: str) -> bool:
        return sid in self._store

    def append(self, sid: str, role: str, text: str) -> None:
        self._store[sid].append({"role": role, "text": text})

    def history(self, sid: str) -> list[dict]:
        return self._store.get(sid, [])

In [3]:
# ── Step 1 tests ──────────────────────────────────────────────────────────────
# Four cases on ChatRequest (happy path + the three validation paths) and one
# round-trip on SessionManager.

try:
    ChatRequest(user_id="alice", message="Hello!")
    print("valid message:        PASS")
except Exception:
    print("valid message:        FAIL")

try:
    r = ChatRequest(user_id="bob", message="A" * 1000)
    print(f"truncation 500:       {'PASS' if len(r.message) == 500 else 'FAIL'} (len={len(r.message)})")
except Exception:
    print("truncation 500:       FAIL")

try:
    ChatRequest(user_id="x", message="   ")
    print("reject empty message: FAIL")
except Exception:
    print("reject empty message: PASS")

try:
    ChatRequest(user_id="", message="hello")
    print("reject empty user_id: FAIL")
except Exception:
    print("reject empty user_id: PASS")

sm = SessionManager()
sid = sm.create()
sm.append(sid, "user", "Hi")
sm.append(sid, "agent", "Welcome!")
print(f"SessionManager:       PASS (session {sid}, {len(sm.history(sid))} messages)")

valid message:        PASS
truncation 500:       PASS (len=500)
reject empty message: PASS
reject empty user_id: PASS
SessionManager:       PASS (session 083049d2, 2 messages)


---
## Step 2 - A safe tool

The tool is `search_product`, a simple keyword search over a static `CATALOG`. The keyword search itself is trivial; the *interesting* part of this step is the production hygiene around it:

- **Input sanitisation.** The `sanitize` helper strips SQL-injection-style metacharacters and caps input length. The tool body here uses a plain Python list, so SQL injection is not literally a threat - but a real `search_product` against a database would be vulnerable, and sanitising at the tool boundary is the cheapest way to keep that surface clean.
- **Environment-variable config.** `DATABASE_URL` is read via `os.getenv` with a safe default. The credential never appears in source code; it lives in `.env` (in development) or in a secret manager (in production). This is exactly the pattern slide 04 (*Gestione Segreti*) walks through.
- **Structured logging.** Every call logs the user id and the sanitised query at INFO level. In production this becomes the audit trail: who searched for what and when. The `logger` instance is the same one configured in step 1, so every component shares the same channel.
- **No hardcoded credentials.** This is what the test at the bottom checks - by inspecting the source of `search_product` and asserting that there is no literal password and that `os.getenv` shows up.

In [4]:
# ── Static catalog (stands in for a real database) ────────────────────────────

CATALOG: list[dict] = [
    {"id": 1, "name": "Laptop Pro 15",              "price": 1299.99},
    {"id": 2, "name": "Ergonomic Mouse",            "price":   34.50},
    {"id": 3, "name": "Mechanical Keyboard",        "price":   89.00},
    {"id": 4, "name": "UltraWide Monitor",          "price":  549.00},
    {"id": 5, "name": "Noise-Cancelling Headphones", "price":  199.90},
    {"id": 6, "name": "HD Webcam",                  "price":   59.90},
]


def sanitize(raw: str) -> str:
    # Strip SQL-injection-style metacharacters and cap length. Cheap belt-and-
    # braces against a database tool we have not written yet but will.
    for ch in ("'", '"', ";", "--", "/*", "*/"):
        raw = raw.replace(ch, "")
    return raw.strip()[:100]


def search_product(query: str, user_id: str) -> dict:
    # Read connection string from env (with a safe default for local dev).
    # The credential never appears in the source.
    db_url = os.getenv("DATABASE_URL", "sqlite:///catalog.db")

    q = sanitize(query)
    logger.info(f"SEARCH | user={user_id} | query='{q}' | db={db_url.split('://')[0]}")

    results = [p for p in CATALOG if q.lower() in p["name"].lower()]
    return {"query": q, "results": results, "total": len(results)}

In [5]:
# ── Step 2 tests ──────────────────────────────────────────────────────────────
# Functional check (the search returns something), security check (sanitisation
# strips a SQL-injection payload), and three static-analysis checks on the
# source code to enforce the hygiene rules.

import inspect
source = inspect.getsource(search_product)

r1 = search_product("laptop", "user_01")
print(f"basic search:         {'PASS' if r1['total'] > 0 else 'FAIL'} ({r1['total']} results)")

r2 = search_product("laptop'; DROP TABLE x; --", "user_01")
stripped = "'" not in r2["query"] and "--" not in r2["query"]
print(f"sanitisation:         {'PASS' if stripped else 'FAIL'}")

print(f"no hardcoded pwd:     {'PASS' if 'supersecret' not in source and 'password=' not in source else 'FAIL'}")
print(f"uses os.getenv:       {'PASS' if 'getenv' in source else 'FAIL'}")
print(f"has logging:          {'PASS' if 'logger' in source else 'FAIL'}")

23:10:36 | INFO | shopassist | SEARCH | user=user_01 | query='laptop' | db=sqlite
23:10:36 | INFO | shopassist | SEARCH | user=user_01 | query='laptop DROP TABLE x' | db=sqlite


basic search:         PASS (1 results)
sanitisation:         PASS
no hardcoded pwd:     PASS
uses os.getenv:       PASS
has logging:          PASS


---
## Step 3 - The production server

FastAPI app with five endpoints, each one doing a single thing:

| Method | Path | Purpose | Failure modes |
|--------|------|---------|---------------|
| POST | `/chat` | Single chat turn, timeout-gated | 504 if the agent takes more than 3 seconds |
| GET | `/history/{session_id}` | Fetch the conversation history for a session | 404 if the session does not exist |
| POST | `/report` | Submit a long-running report job | Returns a `task_id` immediately, never blocks |
| GET | `/report/status/{task_id}` | Poll the status of a report job | 404 if the task does not exist |
| GET | `/health` | Liveness probe for the container orchestrator | Always 200 |

Three design decisions worth flagging.

**`/chat` has a hard timeout via `asyncio.wait_for`.** Three seconds is short on purpose: an interactive chat endpoint must respond fast or the user thinks the service is dead. If the agent legitimately needs longer (because it is doing retrieval, ReAct, multi-step planning), the request is rejected with 504 and the user is told to use a different endpoint. This is *the* canonical pattern for chat APIs: short timeout in the foreground, long jobs on a separate path.

**`/report` does not block.** The handler creates a `task_id`, kicks off a background `asyncio.create_task`, and returns immediately. The client polls `/report/status/{task_id}` to find out when the result is ready. In a real deployment the background work would run on a separate worker (Celery, RQ, an SQS-backed worker) so the API server stays free; in this notebook it lives in the same process for simplicity. The interface stays the same in either case.

**`/health` is dead simple.** A liveness probe should *not* do work. It returns 200 + a fixed JSON blob. If the process is up enough to reply, it is healthy enough to receive traffic. Anything more expensive than that and you risk false negatives (the probe times out under load and the orchestrator kills a healthy pod).

**The `agent` function is a stub.** It returns a synthetic reply with a fake `asyncio.sleep` delay - 5 seconds for messages that look like report requests, 0.3 seconds for everything else. The 5-second variant is what triggers the 504 path in the timeout test. Replacing this stub with a real agent from exercises 02-06 changes nothing in the server.

In [6]:
# ── FastAPI application ───────────────────────────────────────────────────────

app = FastAPI(title="ShopAssist Service")
sessions = SessionManager()
task_store: dict[str, dict] = {}


async def agent(message: str, history: list[dict]) -> str:
    # Stub agent: fake a slow path for report/analysis questions, fast path for
    # everything else. Replace this body with any of the agents from exercises
    # 02 to 06; the rest of the service does not care.
    if "report" in message.lower() or "analysis" in message.lower():
        await asyncio.sleep(5)  # slow path, will trigger the /chat timeout
    else:
        await asyncio.sleep(0.3)

    if "search" in message.lower():
        res = search_product(message.replace("search", "").strip(), "agent")
        if res["total"] > 0:
            return f"Found {res['total']}: {', '.join(p['name'] for p in res['results'])}"
        return "No products found."

    return f"[{len(history)} msgs] reply to: {message}"


@app.post("/chat")
async def chat(req: ChatRequest):
    sid = req.session_id if (req.session_id and sessions.exists(req.session_id)) else sessions.create()
    sessions.append(sid, "user", req.message)
    try:
        reply = await asyncio.wait_for(
            agent(req.message, sessions.history(sid)),
            timeout=3.0,
        )
        sessions.append(sid, "agent", reply)
        return {"reply": reply, "session_id": sid}
    except asyncio.TimeoutError:
        # 504 Gateway Timeout is the right code: WE are an upstream of the agent
        # and the agent did not respond in time. Tell the client to use /report.
        raise HTTPException(
            status_code=504,
            detail="agent did not reply within 3 seconds; use /report for long-running queries.",
        )


@app.get("/history/{session_id}")
async def get_history(session_id: str):
    history = sessions.history(session_id)
    if not history:
        raise HTTPException(404, "session not found")
    return {"session_id": session_id, "messages": history}


async def report_worker(task_id: str, query: str, user_id: str) -> None:
    # Simulated long-running job. In production this would live in a separate
    # worker process (Celery, RQ) so the API server can stay free.
    task_store[task_id]["status"] = "processing"
    await asyncio.sleep(3)
    task_store[task_id].update({
        "status": "completed",
        "result": search_product(query, user_id),
    })


@app.post("/report")
async def submit_report(req: ChatRequest):
    task_id = str(uuid.uuid4())[:8]
    task_store[task_id] = {"status": "queued", "result": None}
    asyncio.create_task(report_worker(task_id, req.message, req.user_id))
    return {"task_id": task_id}


@app.get("/report/status/{task_id}")
async def report_status(task_id: str):
    if task_id not in task_store:
        raise HTTPException(404, "task not found")
    return task_store[task_id]


@app.get("/health")
async def health():
    # Liveness probe: cheap, deterministic, no I/O. The orchestrator hits this
    # every few seconds; anything heavy here invents false negatives under load.
    return {"status": "ok"}

In [7]:
# ── Start the server in a background thread ──────────────────────────────────
# Jupyter-specific accommodation so the test cell below can hit the server
# over real HTTP. In production this notebook is irrelevant; you run uvicorn
# from the container's CMD line instead (see Dockerfile in step 4).
#
# If you re-run this cell after a previous run, the daemon thread from the
# first run still owns port 8050 and uvicorn will fail to bind. Restart the
# kernel between runs to free the port.

PORT = 8050

threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"),
    daemon=True,
).start()
time.sleep(1)  # give uvicorn a moment to bind the port

print(f"Server up on http://127.0.0.1:{PORT}")

Server up on http://127.0.0.1:8050


In [8]:
# ── Step 3 end-to-end test ────────────────────────────────────────────────────
# Four checks: session continuity, timeout path, async-task submission,
# async-task polling.

async def test_server():
    async with httpx.AsyncClient(base_url=f"http://127.0.0.1:{PORT}", timeout=10) as client:
        # 1. Two-turn conversation. The second turn should land in the same
        #    session as the first, so the history grows to four messages.
        r1 = await client.post("/chat", json={"user_id": "alice", "message": "Hi"})
        sid = r1.json()["session_id"]
        await client.post("/chat", json={"user_id": "alice", "session_id": sid, "message": "search laptop"})
        h = await client.get(f"/history/{sid}")
        n = len(h.json()["messages"])
        print(f"sessions:        {'PASS' if n >= 4 else 'FAIL'} ({n} messages)")

        # 2. Timeout path. 'report' triggers the 5-second slow path; the 3-second
        #    /chat timeout should fire and return 504.
        start = time.time()
        r2 = await client.post("/chat", json={"user_id": "bob", "message": "give me a report"})
        elapsed = time.time() - start
        ok = r2.status_code == 504 and 2.9 < elapsed < 3.5
        print(f"timeout 504:     {'PASS' if ok else 'FAIL'} (status={r2.status_code}, elapsed={elapsed:.1f}s)")

        # 3. Async report submission. Should return a task_id in well under a
        #    second, regardless of how long the actual job will take.
        start = time.time()
        r3 = await client.post("/report", json={"user_id": "carl", "message": "laptop"})
        elapsed = time.time() - start
        body = r3.json()
        ok = elapsed < 1.0 and "task_id" in body
        print(f"report submit:   {'PASS' if ok else 'FAIL'} ({elapsed:.2f}s, body keys: {list(body)})")

        # 4. Poll the status endpoint until the job completes (or we give up).
        if "task_id" in body:
            task_id = body["task_id"]
            for i in range(6):
                await asyncio.sleep(1)
                status = (await client.get(f"/report/status/{task_id}")).json()
                if status.get("status") == "completed":
                    n_results = status["result"]["total"]
                    print(f"report poll:     PASS (completed after {i + 1}s, {n_results} results)")
                    break
            else:
                print("report poll:     FAIL (still not completed after 6s)")

        # 5. Health probe.
        h = await client.get("/health")
        print(f"health probe:    {'PASS' if h.status_code == 200 and h.json() == {'status': 'ok'} else 'FAIL'}")


await test_server()

23:10:38 | INFO | httpx | HTTP Request: POST http://127.0.0.1:8050/chat "HTTP/1.1 200 OK"
23:10:38 | INFO | shopassist | SEARCH | user=agent | query='laptop' | db=sqlite
23:10:38 | INFO | httpx | HTTP Request: POST http://127.0.0.1:8050/chat "HTTP/1.1 200 OK"
23:10:38 | INFO | httpx | HTTP Request: GET http://127.0.0.1:8050/history/d6377b0e "HTTP/1.1 200 OK"


sessions:        PASS (4 messages)


23:10:41 | INFO | httpx | HTTP Request: POST http://127.0.0.1:8050/chat "HTTP/1.1 504 Gateway Timeout"
23:10:41 | INFO | httpx | HTTP Request: POST http://127.0.0.1:8050/report "HTTP/1.1 200 OK"


timeout 504:     PASS (status=504, elapsed=3.0s)
report submit:   PASS (0.01s, body keys: ['task_id'])


23:10:42 | INFO | httpx | HTTP Request: GET http://127.0.0.1:8050/report/status/5f0a6a59 "HTTP/1.1 200 OK"
23:10:43 | INFO | httpx | HTTP Request: GET http://127.0.0.1:8050/report/status/5f0a6a59 "HTTP/1.1 200 OK"
23:10:44 | INFO | shopassist | SEARCH | user=carl | query='laptop' | db=sqlite
23:10:44 | INFO | httpx | HTTP Request: GET http://127.0.0.1:8050/report/status/5f0a6a59 "HTTP/1.1 200 OK"
23:10:44 | INFO | httpx | HTTP Request: GET http://127.0.0.1:8050/health "HTTP/1.1 200 OK"


report poll:     PASS (completed after 3s, 1 results)
health probe:    PASS


---
## Step 4 - Containerisation

Six artefacts written to a `shopassist_deploy/` directory. None of them are valuable in isolation; the *combination* is what lets the service ship.

| File | Purpose |
|------|---------|
| `Dockerfile` | Builds a reproducible Python 3.10 image with the service and its dependencies. |
| `requirements.txt` | Pinned versions of the runtime dependencies. Pinning is what makes the image reproducible. |
| `agent.py` | Service entry point (stub here; in a real build it would import the real agent). |
| `.env` | Local-only environment variables. **Must** be gitignored. |
| `.env.example` | Documents what variables `.env` should contain. Safe to commit. |
| `.gitignore` | Prevents `.env` (and other local-only files) from being committed. |
| `docker-compose.yml` | One-command local deploy: builds the image, mounts `.env`, opens port 8000, configures a healthcheck. |

**The placeholder in `.env`.** The file generated below contains `OPENAI_API_KEY=replace-this-with-your-key`. That string is *deliberately* not in the `sk-...` shape: it should not survive accidental copy-paste into a script that expects a real key, and it cannot be mistaken for a leaked secret in an audit scan. The point of writing this file is to demonstrate the *file-and-gitignore pattern*, not to ship a credential.

**The `shopassist_deploy/` directory itself.** This directory is generated by running the cell. It is NOT part of the notebook source; it is a build artefact. If you commit your notebook with this directory next to it, the directory ends up in git too - which is fine for `Dockerfile` / `.env.example` / `docker-compose.yml` but NOT for `.env`. Either delete the directory after running the cell (`rm -rf shopassist_deploy/`) or add `shopassist_deploy/.env` to a top-level `.gitignore` before committing.

In [9]:
# ── Generate the deployment package ───────────────────────────────────────────

out = Path("shopassist_deploy")
out.mkdir(exist_ok=True)

# Pinned requirements. Real version pins matter: an unpinned build is not
# reproducible, and pip will eventually resolve to a new version that breaks
# the image at the worst possible moment.
(out / "requirements.txt").write_text(
    "fastapi==0.109.0\n"
    "uvicorn==0.27.0\n"
    "pydantic==2.5.0\n"
)

# Stub agent.py. In a real build this would import the agents from
# exercises 02 to 06; the deployment scaffolding around it stays the same.
(out / "agent.py").write_text(
    "import os\n"
    "from fastapi import FastAPI\n\n"
    "app = FastAPI(title='ShopAssist Service')\n\n"
    "@app.get('/health')\n"
    "async def health():\n"
    "    return {'status': 'ok', 'env': os.getenv('LOG_LEVEL', 'info')}\n"
)

# Local-only env file. The KEY is intentionally NOT a real-looking 'sk-...'
# placeholder - we never want a string that looks like a leaked secret to
# land in the working directory of a course repo.
(out / ".env").write_text(
    "# Local development overrides. NEVER commit this file.\n"
    "OPENAI_API_KEY=replace-this-with-your-key\n"
    "LOG_LEVEL=info\n"
)

# .env.example documents the schema without carrying secrets. Safe to commit.
(out / ".env.example").write_text(
    "# Copy this file to .env and fill in real values.\n"
    "OPENAI_API_KEY=your-key-here\n"
    "LOG_LEVEL=info\n"
)

# .gitignore protects .env and other local-only files. The bang on
# !.env.example explicitly re-includes the example file so it does ship.
(out / ".gitignore").write_text(
    ".env\n"
    ".env.*\n"
    "!.env.example\n"
    "__pycache__/\n"
    "*.pyc\n"
)

# Dockerfile. Single-stage, slim base image. Production-grade Dockerfiles
# would add a non-root USER, multi-stage build to drop pip from the final
# image, and a more thoughtful CMD with --workers. Discussed in the
# critical analysis section below.
(out / "Dockerfile").write_text(
    "FROM python:3.10-slim\n"
    "WORKDIR /app\n"
    "COPY requirements.txt .\n"
    "RUN pip install --no-cache-dir -r requirements.txt\n"
    "COPY . .\n"
    "EXPOSE 8000\n"
    'CMD ["uvicorn", "agent:app", "--host", "0.0.0.0", "--port", "8000"]\n'
)

# docker-compose.yml: one-command local deploy. The healthcheck wires the
# /health endpoint into the container runtime so an unhealthy container can
# be restarted automatically.
(out / "docker-compose.yml").write_text(
    'version: "3.8"\n'
    "services:\n"
    "  shopassist:\n"
    "    build: .\n"
    '    ports: ["8000:8000"]\n'
    "    env_file: [.env]\n"
    "    restart: unless-stopped\n"
    "    healthcheck:\n"
    '      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]\n'
    "      interval: 30s\n"
    "      timeout: 5s\n"
    "      retries: 3\n"
)

print(f"Wrote deployment package to {out.resolve()}")
for path in sorted(out.iterdir()):
    print(f"  {path.name}")

Wrote deployment package to /Users/simone/github/AI_solutions_architect_course/03_agentic_ai/exercises/shopassist_deploy
  .env
  .env.example
  .gitignore
  Dockerfile
  agent.py
  docker-compose.yml
  requirements.txt


In [10]:
# ── Step 4 tests ──────────────────────────────────────────────────────────────
# Static checks on the seven files we just wrote.

dockerfile  = (out / "Dockerfile").read_text()
gitignore   = (out / ".gitignore").read_text()
compose     = (out / "docker-compose.yml").read_text()
env_example = (out / ".env.example").read_text()

checks = [
    ("WORKDIR" in dockerfile,                                   "Dockerfile WORKDIR"),
    ("EXPOSE"  in dockerfile,                                   "Dockerfile EXPOSE"),
    ("requirements.txt" in dockerfile and "RUN pip" in dockerfile, "Dockerfile pip install"),
    ("--host"  in dockerfile and "--port"  in dockerfile,       "Dockerfile CMD complete"),
    (".env"    in gitignore,                                    ".gitignore protects .env"),
    ("!.env.example" in gitignore,                              ".gitignore allows .env.example"),
    ((out / ".env.example").exists() and env_example.strip() != "", ".env.example present"),
    ("env_file"    in compose,                                  "docker-compose uses env_file"),
    ("healthcheck" in compose,                                  "docker-compose has healthcheck"),
]

score = 0
for ok, label in checks:
    score += int(ok)
    print(f"  {label:32s} {'PASS' if ok else 'FAIL'}")

print(f"\nStep 4 score: {score}/{len(checks)}")

  Dockerfile WORKDIR               PASS
  Dockerfile EXPOSE                PASS
  Dockerfile pip install           PASS
  Dockerfile CMD complete          PASS
  .gitignore protects .env         PASS
  .gitignore allows .env.example   PASS
  .env.example present             PASS
  docker-compose uses env_file     PASS
  docker-compose has healthcheck   PASS

Step 4 score: 9/9


---
## 5. Critical analysis

### What the deployment wrapper actually buys

Exercises 01-06 produced agents that worked in a notebook. This one wraps an agent function in five concrete production primitives: input validation at the request boundary, session state, a tool layer with sanitisation and structured logging, a server with timeouts and background jobs, and a container with proper secret hygiene. None of those primitives is hard individually, and the LLM does not appear anywhere in the wrapper - that is the point. The agent is a single function call deep inside `/chat`; everything else exists to make that single call survive contact with real users, real networks, and real ops teams.

### Stateless vs stateful, in practice

The default HTTP service is stateless: each request stands on its own, any instance can answer any request, scaling out is trivial. As soon as we add the `SessionManager`, that property breaks: a follow-up request to a session has to land on the *same* instance that holds the session, or it gets a fresh empty history. The in-memory `SessionManager` here is fine for a single-instance demo and disastrous for anything else. Three production fixes, in order of cost:

- **Sticky sessions on the load balancer.** Cheap, but turns a stateless deployment into a stateful one and complicates rolling deploys.
- **External session store (Redis, sqlite WAL, a database).** The cleanest fix: the service stays stateless, the *state* moves to a tier that can be scaled and replicated independently.
- **Stateful service with a persistent volume.** Useful when sessions are large and stateful work dominates; rare in chat services.

The choice is mostly about how much load the service has to handle. For ten concurrent users, an in-memory store with a periodic dump to disk is enough. For ten thousand, you want Redis.

### Three deployment patterns that earn their cost

**The 3-second `/chat` timeout.** Without it, a slow agent ties up a uvicorn worker; with enough concurrent slow requests the service stops responding to fast ones. The timeout is a circuit breaker that keeps the foreground responsive. Three seconds is on the aggressive side; a real deployment would tune it based on the p99 latency of the agent under the actual load.

**Async background tasks for long jobs.** The `/report` pattern looks heavier than it is: one extra endpoint to submit, one to poll, a process-local task store. In exchange the foreground stays fast and the background can run for as long as it needs. For real deployments the task store moves out of the API process (Redis, a database, a message queue) so it survives restarts. The interface that the client sees - `task_id` plus polling - does not change.

**Cheap health endpoint.** `/health` here returns a fixed dict. A tempting mistake is to make `/health` do something useful, e.g. check the database connection or run a tool. Resist: under load, that health check times out, the orchestrator concludes the pod is dead, and starts killing healthy pods. Cheap-and-dumb is a feature.

### Secret management: the floor and the ceiling

The `.env` + `.gitignore` + `.env.example` triple is the floor. It is what every solo developer should do on day one, and it is what every team should require in the project bootstrap. It is not the ceiling.

In production:

- `.env` is replaced by **a real secret manager** (AWS Secrets Manager, GCP Secret Manager, HashiCorp Vault). The container does not see the raw secret at build time; it fetches it at startup or per-request via an IAM-authenticated call.
- The deployment pipeline never has the secret in any text file. Build artefacts and logs do not contain the secret. Compromised CI does not leak production credentials.
- Secrets are rotated automatically. The application reloads them without a restart.

The lesson is that the file-and-gitignore pattern is necessary but not sufficient. Treat it as a *baseline*: if the project is missing it, fix it first; if the project has it but the production system also has the same approach, fix it again.

### What this Dockerfile is missing for production

Four things this Dockerfile leaves on the table. Each is worth one extra line and a measurable improvement.

- **Non-root user.** `USER appuser` before `CMD`. A compromised container with root inside is worse than the same container with an unprivileged user.
- **Multi-stage build.** Run pip in a `builder` stage, copy only the installed packages into a `runtime` stage. Smaller image, fewer attack surfaces (pip itself, build-essential).
- **`--workers` on uvicorn.** A single-worker FastAPI server runs all requests on one thread. For CPU-bound paths (anything that briefly blocks the loop) that hurts under load. `uvicorn agent:app --workers 4` is the easy fix; gunicorn with uvicorn workers is the more robust one.
- **Healthcheck with backoff.** The compose file uses a flat 30-second interval. A startup probe with shorter intervals during the first minute (when the service is still booting) and a longer interval after that catches start-up failures faster.

None of these is strictly necessary for a working demo; all four are standard for anything that takes traffic.

### Looking back at module 03

Seven exercises, one arc.

- Ex 01: workflow versus autonomous agent (the orchestration-by-human end of the spectrum).
- Ex 02: tool-calling agent with two real tools (the orchestration-by-LLM end).
- Ex 03: ReAct + Reflexion - the agent critiquing itself.
- Ex 04: hand-rolled versus framework, LangChain versus LangGraph.
- Ex 05: short-term memory, trimming versus summarisation.
- Ex 06: long-term memory, RAG + CAG + episodic state.
- Ex 07: the deployment wrapper - putting an agent behind an HTTP endpoint and in a container.

The progression is intentional: the early exercises establish the vocabulary (workflow, agent, tool, ReAct, Reflexion), the middle ones add the structural layers (frameworks, memory), and this last one shows that everything we built fits in a single function inside a service that someone else can run. If there is one thread that runs through all seven, it is the *separation of concerns*: at every step, deciding what the LLM does versus what plain Python does. The deployment wrapper here is the rightmost end of that line - the model only sees what it absolutely has to, the rest is just engineering.